<a href="https://colab.research.google.com/github/MarcelinaBytes/readmission-prediction-ML/blob/main/notebooks/03_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Modeling

In [41]:
#Repo root

%cd /content/readmission-prediction-ML
import os
os.getcwd()

/content/readmission-prediction-ML


'/content/readmission-prediction-ML'

In [42]:
#import my libraries

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

import joblib
import glob

In [43]:
#load clean dataset

df = pd.read_csv('data/processed/clean_step1.csv')
df.shape, df.head()

((101766, 48),
    encounter_id  patient_nbr             race  gender      age  \
 0       2278392      8222157        Caucasian  Female   [0-10)   
 1        149190     55629189        Caucasian  Female  [10-20)   
 2         64410     86047875  AfricanAmerican  Female  [20-30)   
 3        500364     82442376        Caucasian    Male  [30-40)   
 4         16680     42519267        Caucasian    Male  [40-50)   
 
    admission_type_id  discharge_disposition_id  admission_source_id  \
 0                  6                        25                    1   
 1                  1                         1                    7   
 2                  1                         1                    7   
 3                  1                         1                    7   
 4                  1                         1                    7   
 
    time_in_hospital  num_lab_procedures  ...  insulin  glyburide-metformin  \
 0                 1                  41  ...       No              

In [44]:
#define target and features

y = df['readmitted_flag'].astype(int)

# Numerical features
num_features = [
    col for col in [
        'time_in_hospital', 'num_lab_procedures', 'num_procedures',
        'num_medications', 'number_outpatient','number_emergency',
        'number_inpatient', 'prior_utilization', 'a1c_score'
    ] if col in df.columns
]

# Include diagnosis counts if present
num_features += [c for c in df.columns if c.startswith('diag_count_')]

# Categorical features
cat_features = [
    col for col in [
        'gender','age','admission_type_id','discharge_disposition_id',
        'admission_source_id','insulin'
    ] if col in df.columns
]

# Final feature matrix
feature_cols = num_features + cat_features
X = df[feature_cols].copy()

X.head(), len(feature_cols)

(   time_in_hospital  num_lab_procedures  num_procedures  num_medications  \
 0                 1                  41               0                1   
 1                 3                  59               0               18   
 2                 2                  11               5               13   
 3                 2                  44               1               16   
 4                 1                  51               0                8   
 
    number_outpatient  number_emergency  number_inpatient  gender      age  \
 0                  0                 0                 0  Female   [0-10)   
 1                  0                 0                 0  Female  [10-20)   
 2                  2                 0                 1  Female  [20-30)   
 3                  0                 0                 0    Male  [30-40)   
 4                  0                 0                 0    Male  [40-50)   
 
    admission_type_id  discharge_disposition_id  admission_source_

In [45]:
#train test and split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.shape, X_test.shape

((81412, 13), (20354, 13))

In [46]:
#build preprocessing pipeline

numeric_tf = Pipeline(steps=[
    ("scaler", StandardScaler(with_mean=False))
])

categorical_tf = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=True
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_tf, num_features),
        ("cat", categorical_tf, cat_features)
    ],
    sparse_threshold=0.3
)

In [48]:
#Model 1- run a Logistic Regression for a baseline

model_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=500, class_weight='balanced'))
])

model_lr.fit(X_train, y_train)

proba_lr = model_lr.predict_proba(X_test)[:, 1]

roc_lr = roc_auc_score(y_test, proba_lr)
pr_lr  = average_precision_score(y_test, proba_lr)

print(f"LogReg → ROC-AUC: {roc_lr:.3f} | PR-AUC: {pr_lr:.3f}")

LogReg → ROC-AUC: 0.674 | PR-AUC: 0.219


In [49]:
#Model 2- Random Forest

model_rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ))
])

model_rf.fit(X_train, y_train)
proba_rf = model_rf.predict_proba(X_test)[:, 1]

roc_rf = roc_auc_score(y_test, proba_rf)
pr_rf  = average_precision_score(y_test, proba_rf)

print(f"RandomForest → ROC-AUC: {roc_rf:.3f} | PR-AUC: {pr_rf:.3f}")

RandomForest → ROC-AUC: 0.637 | PR-AUC: 0.181


In [50]:
#Model 3- XGBoost

scale_pos = y_train.value_counts()[0] / y_train.value_counts()[1]

model_xgb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos,
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1
    ))
])

model_xgb.fit(X_train, y_train)
proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

roc_xgb = roc_auc_score(y_test, proba_xgb)
pr_xgb  = average_precision_score(y_test, proba_xgb)

print(f"XGBoost → ROC-AUC: {roc_xgb:.3f} | PR-AUC: {pr_xgb:.3f}")

XGBoost → ROC-AUC: 0.682 | PR-AUC: 0.231


In [51]:
import pandas as pd

scores = pd.DataFrame({
    "Model": ["LogReg", "RandomForest", "XGBoost"],
    "ROC_AUC": [roc_lr, roc_rf, roc_xgb],
    "PR_AUC": [pr_lr, pr_rf, pr_xgb]
}).sort_values("PR_AUC", ascending=False)

scores

,Model,ROC_AUC,PR_AUC
2,XGBoost,0.682028,0.231227
0,LogReg,0.673891,0.219318
1,RandomForest,0.637208,0.180668


In [53]:
#Save the best model

best_model = model_xgb   # or model_rf, or model_lr

os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/best_readmission_model.pkl')

print("Saved → models/best_readmission_model.pkl")

Saved → models/best_readmission_model.pkl
